# Day 10 · Qwen3-VL-4B grounding · Kaggle GPU

Resume E5 from the completed E0–E4 results in Kaggle Version 3. Attach that version's output as Notebook Input, and keep the road-damage dataset input attached. E3's trained adapter and projector are restored; the notebook runs E5, scores it, and exports evidence. No test score is entered manually.


In [1]:
import json, os, shutil, subprocess, sys
from pathlib import Path
REPO = "https://github.com/vminhquan/K4-Track-4-Day10-VoMinhQuan-2A202602429-VisionLanguageModel.git"
ROOT = Path("/kaggle/working/day10")
if ROOT.exists():
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO, str(ROOT)], check=True)
os.chdir(ROOT)
print("Working in", Path.cwd())

import kagglehub
PREVIOUS = Path(kagglehub.notebook_output_download(
    "minhquanvo/notebook3f3cc8f9bf/versions/3")) / "day10"
print("Version 3 output mounted at", PREVIOUS)
if not (PREVIOUS / "report/evidence/e4_data_0.1.json").is_file():
    raise FileNotFoundError("Attach completed Version 3 output as Notebook Input before running")
Path("report/evidence").mkdir(parents=True, exist_ok=True)
for source in (PREVIOUS / "report/evidence").glob("*.json"):
    shutil.copy2(source, Path("report/evidence") / source.name)
Path("work").mkdir(exist_ok=True)
shutil.copytree(PREVIOUS / "work/e3", Path("work/e3"), dirs_exist_ok=True)
for source in (PREVIOUS / "work").glob("*_test.jsonl"):
    shutil.copy2(source, Path("work") / source.name)
for name in ("e1", "e2", "e4_data_0.1"):
    source = PREVIOUS / "work" / name / "run.json"
    if source.is_file():
        target = Path("work") / name / "run.json"
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, target)
print("Restored E0–E4 evidence, raw predictions, and E3 checkpoint from Version 3")

def run(*args):
    command = [sys.executable, *map(str, args)]
    print("$", " ".join(command), flush=True)
    subprocess.run(command, check=True)


Cloning into '/kaggle/working/day10'...


Working in /kaggle/working/day10
Version 3 output mounted at /kaggle/input/notebooks/minhquanvo/notebook3f3cc8f9bf/day10
Restored E0–E4 evidence, raw predictions, and E3 checkpoint from Version 3


## 0. Install and validate the examiner tools

Run after enabling GPU. Package installation may require a kernel restart on Kaggle. Rerun this notebook from the first cell after a restart; `data/` and completed checkpoints remain in `/kaggle/working` for the current session.


In [2]:
run("-m", "pip", "install", "-q", "-r", "requirements-colab.txt", "pytest")
# Kaggle ships torchao 0.10; PEFT 0.21.2 rejects it while injecting ViT LoRA.
# This workflow uses bitsandbytes NF4, not torchao quantization.
run("-m", "pip", "uninstall", "-y", "torchao")
run("-m", "pytest", "-q")
run("-m", "unittest", "-v", "test_bonus_reward.py")
run("-m", "unittest", "-v", "test_student_workflow.py")


$ /usr/bin/python3 -m pip install -q -r requirements-colab.txt pytest
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 91.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 42.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 29.9 MB/s eta 0:00:00
$ /usr/bin/python3 -m pip uninstall -y torchao
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
$ /usr/bin/python3 -m pytest -q
...........................s.........................................    [100%]
68 passed, 1 skipped in 0.41s
$ /usr/bin/python3 -m unittest -v test_bonus_reward.py
$ /usr/bin/python3 -m unittest -v test_student_workflow.py


test_both_empty (test_bonus_reward.BonusRewardTests.test_both_empty) ... ok
test_extra_box (test_bonus_reward.BonusRewardTests.test_extra_box) ... ok
test_invalid_json (test_bonus_reward.BonusRewardTests.test_invalid_json) ... ok
test_perfect (test_bonus_reward.BonusRewardTests.test_perfect) ... ok
test_trl_batch (test_bonus_reward.BonusRewardTests.test_trl_batch) ... ok
test_wrong_class (test_bonus_reward.BonusRewardTests.test_wrong_class) ... ok

----------------------------------------------------------------------
Ran 6 tests in 0.001s

OK
test_clipping (test_student_workflow.GroundingLabelTests.test_clipping) ... ok
test_order_and_empty (test_student_workflow.GroundingLabelTests.test_order_and_empty) ... ok
test_thin_box_remains_valid_after_rounding (test_student_workflow.GroundingLabelTests.test_thin_box_remains_valid_after_rounding) ... ok
test_yolo_to_norm1000 (test_student_workflow.GroundingLabelTests.test_yolo_to_norm1000) ... ok

---------------------------------------------

## 1. Prepare the fixed session split

Attach `lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes` via **Add Input**. The examiner's script finds its `images/` and `labels-YOLO/` folder under `/kaggle/input`.


In [3]:
if not Path("data/splits/meta.json").exists():
    run("scripts/prepare_data.py", "--source", "/kaggle/input")
run("lab_workflow.py", "check-data")
version = json.loads(Path("data/splits/meta.json").read_text())["dataset_version"]
for name in ("e0", "e1", "e2", "e3", "e4_data_0.1", "e4_data_1.0"):
    evidence = Path("report/evidence") / f"{name}.json"
    assert evidence.is_file(), name
    assert json.loads(evidence.read_text())["dataset_version"] == version, name
run("lab_workflow.py", "gt-figures")
run("lab_workflow.py", "preflight")


$ /usr/bin/python3 scripts/prepare_data.py --source /kaggle/input
dataset_version : d10-21e4070301
  train  1422 ảnh  boxes={'pothole': 1074, 'crack': 1750, 'manhole': 691}
  val     276 ảnh  boxes={'pothole': 66, 'crack': 470, 'manhole': 103}
  test    311 ảnh  boxes={'pothole': 121, 'crack': 299, 'manhole': 163}
$ /usr/bin/python3 lab_workflow.py check-data
{
  "dataset_version": "d10-21e4070301",
  "splits": {
    "train": {
      "images": 1422,
      "boxes": {
        "manhole": 691,
        "pothole": 1074,
        "crack": 1750
      }
    },
    "val": {
      "images": 276,
      "boxes": {
        "crack": 470,
        "manhole": 103,
        "pothole": 66
      }
    },
    "test": {
      "images": 311,
      "boxes": {
        "crack": 299,
        "manhole": 163,
        "pothole": 121
      }
    }
  }
}
$ /usr/bin/python3 lab_workflow.py gt-figures
report/figures/gt_01_20250216_164325.png
report/figures/gt_02_vlcsnap-00137.png
report/figures/gt_03_vlcsnap-2025-02-19-13

Loading weights: 100%|██████████| 713/713 [00:07<00:00, 94.55it/s] 


Masked answer: [{"label":"crack","bbox_2d":[430,467,583,506]},{"label":"pothole","bbox_2d":[361,475,414,522]},{"label":"manhole","bbox_2d":[317,481,616,625]},{"label":"pothole","bbox_2d":[583,500,627,614]},{"label":"crack","bbox_2d":[580,508,655,614]},{"label":"pothole","bbox_2d":[259,517,311,572]},{"label":"crack
Vision tokens: 220; image_grid_thw=[[1, 22, 40]]


## 2. E0 zero-shot on the full 311-image test split

Save the raw predictions in `work/e0_test.jsonl`; the scoring script writes the evidence JSON.


In [4]:
assert Path("report/evidence/e0.json").is_file()
print("E0 restored from Version 3")


E0 restored from Version 3


## 3. E1: LoRA on the LLM only

A fresh Python process runs each stage, so VRAM is released before evaluation. Each SFT run saves an adapter and `run.json` every 100 steps. If Kaggle reports OOM, use `--batch-size 1 --grad-accum 8` in the train call; effective batch remains 8.


In [5]:
assert Path("report/evidence/e1.json").is_file()
print("E1 restored from Version 3")


E1 restored from Version 3


## 4. E2: LLM LoRA + full float32 projector

The projector is saved separately as `work/e2/projector.pt` and reloaded for inference.


In [6]:
assert Path("report/evidence/e2.json").is_file()
print("E2 restored from Version 3")


E2 restored from Version 3


## 5. E3: E2 + ViT LoRA

Expected LoRA target counts: E1/E2 = 252, E3 = 348. The code stops if the model module tree differs.


In [7]:
assert all(Path(p).exists() for p in (
    "report/evidence/e3.json", "work/e3/run.json", "work/e3/adapter",
    "work/e3/projector.pt", "work/e3_test.jsonl"))
print("E3 restored from Version 3")
run("lab_workflow.py", "results")


E3 restored from Version 3
$ /usr/bin/python3 lab_workflow.py results
e0              F1=0.0986 CI95=[0.07042253521126762, 0.12881355932203392] valid=0.961 FP/img=0.791
e1              F1=0.2724 CI95=[0.23634735899731424, 0.30901287553648066] valid=0.997 FP/img=1.225
e2              F1=0.3324 CI95=[0.29246935201401053, 0.37222222222222223] valid=1.000 FP/img=1.077
e3              F1=0.4367 CI95=[0.3978102189781022, 0.47782805429864256] valid=1.000 FP/img=0.945
e4_data_0.1     F1=0.2720 CI95=[0.2401171303074671, 0.30702389572773353] valid=1.000 FP/img=2.006
e4_data_1.0     F1=0.4367 CI95=[0.3978102189781022, 0.47782805429864256] valid=1.000 FP/img=0.945


## 6. E4: clean ablation of train data size

Pick the best E1–E3 by test F1. The 100% condition reuses its existing checkpoint and reruns inference/scoring; the 10% condition retrains from the same base model with every other setting unchanged. The fixed 10% sample uses seed 42. Both evidence files are generated by `score.py`.


In [8]:
assert all(Path(p).is_file() for p in (
    "report/evidence/e4_data_0.1.json", "report/evidence/e4_data_1.0.json",
    "work/e4_data_0.1_test.jsonl", "work/e4_data_1.0_test.jsonl"))
print("Both E4 conditions restored from Version 3")
run("lab_workflow.py", "results")


Both E4 conditions restored from Version 3
$ /usr/bin/python3 lab_workflow.py results
e0              F1=0.0986 CI95=[0.07042253521126762, 0.12881355932203392] valid=0.961 FP/img=0.791
e1              F1=0.2724 CI95=[0.23634735899731424, 0.30901287553648066] valid=0.997 FP/img=1.225
e2              F1=0.3324 CI95=[0.29246935201401053, 0.37222222222222223] valid=1.000 FP/img=1.077
e3              F1=0.4367 CI95=[0.3978102189781022, 0.47782805429864256] valid=1.000 FP/img=0.945
e4_data_0.1     F1=0.2720 CI95=[0.2401171303074671, 0.30702389572773353] valid=1.000 FP/img=2.006
e4_data_1.0     F1=0.4367 CI95=[0.3978102189781022, 0.47782805429864256] valid=1.000 FP/img=0.945


## 7. Bonus E5: GRPO with verifiable IoU reward

The reward has six independent CPU tests. GRPO resumes from the trained E3 adapter, generates four completions per image, trains for 60 steps at 5e-6 and saves its own adapter/projector and log. The full projector stays float32; generation uses fp16 autocast to match the Qwen3-VL vision tower.


In [9]:
run("-m", "unittest", "-v", "test_bonus_reward.py")
run("lab_workflow.py", "grpo")
run("lab_workflow.py", "predict", "--name", "e5")
run("lab_workflow.py", "score", "--name", "e5")
run("lab_workflow.py", "results")


$ /usr/bin/python3 -m unittest -v test_bonus_reward.py
$ /usr/bin/python3 lab_workflow.py grpo


test_both_empty (test_bonus_reward.BonusRewardTests.test_both_empty) ... ok
test_extra_box (test_bonus_reward.BonusRewardTests.test_extra_box) ... ok
test_invalid_json (test_bonus_reward.BonusRewardTests.test_invalid_json) ... ok
test_perfect (test_bonus_reward.BonusRewardTests.test_perfect) ... ok
test_trl_batch (test_bonus_reward.BonusRewardTests.test_trl_batch) ... ok
test_wrong_class (test_bonus_reward.BonusRewardTests.test_wrong_class) ... ok

----------------------------------------------------------------------
Ran 6 tests in 0.001s

OK
Loading weights: 100%|██████████| 713/713 [00:07<00:00, 96.79it/s] 


GRPO float32 trainable: 148426752; projector: 109105152


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 151643}.
100%|██████████| 60/60 [27:33<00:00, 25.53s/it]

{'loss': '0.07295', 'grad_norm': 'nan', 'learning_rate': '4.667e-06', 'num_tokens': '1.532e+04', 'completions/mean_length': '58.98', 'completions/min_length': '34.6', 'completions/max_length': '82', 'completions/clipped_ratio': '0.05', 'completions/mean_terminated_length': '55.7', 'completions/min_terminated_length': '34.6', 'completions/max_terminated_length': '78.8', 'rewards/trl_reward/mean': '0.3816', 'rewards/trl_reward/std': '0.1422', 'reward': '0.3816', 'reward_std': '0.1422', 'frac_reward_zero_std': '0', 'entropy': '0.487', 'clip_ratio/low_mean': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/region_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_max': '0', 'step_time': '25.31', 'epoch': '0.007032'}
{'loss': '0.03488', 'grad_norm': '1.706', 'learning_rate': '4.25e-06', 'num_tokens': '3.094e+04', 'completions/mean_length': '66.55', 'completions/min_length': '34.4', 'completions/max_length': '103.4', 'completions/clipped_ratio': '0.075', 'completions/mean_terminated_length

...

e5: 240/311


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transform

e5: 280/311


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transform

$ /usr/bin/python3 lab_workflow.py score --name e5
E5 [test] n=311 valid=1.000 P=0.478 R=0.425 F1=0.450 CI95=0.411-0.492 meanIoU=0.690 FP/img=0.87
   pothole  P=0.385 R=0.306 F1=0.341
   crack    P=0.359 R=0.341 F1=0.350
   manhole  P=0.784 R=0.669 F1=0.722
$ /usr/bin/python3 lab_workflow.py results
e0              F1=0.0986 CI95=[0.07042253521126762, 0.12881355932203392] valid=0.961 FP/img=0.791
e1              F1=0.2724 CI95=[0.23634735899731424, 0.30901287553648066] valid=0.997 FP/img=1.225
e2              F1=0.3324 CI95=[0.29246935201401053, 0.37222222222222223] valid=1.000 FP/img=1.077
e3              F1=0.4367 CI95=[0.3978102189781022, 0.47782805429864256] valid=1.000 FP/img=0.945
e4_data_0.1     F1=0.2720 CI95=[0.2401171303074671, 0.30702389572773353] valid=1.000 FP/img=2.006
e4_data_1.0     F1=0.4367 CI95=[0.3978102189781022, 0.47782805429864256] valid=1.000 FP/img=0.945
e5              F1=0.4501 CI95=[0.41069887834339946, 0.4916420845624386] valid=1.000 FP/img=0.871


## 8. Visual evidence and return package

Create six paired GT/prediction figures from E3. Export the scored evidence, figures, logs and raw predictions for the final report. The ZIP excludes images, model weights and API tokens.


In [10]:
best = subprocess.check_output([sys.executable, "lab_workflow.py", "best"], text=True).strip()
run("lab_workflow.py", "figures", "--name", best)
subprocess.run([sys.executable, "scripts/check_submission.py"], check=False)


$ /usr/bin/python3 lab_workflow.py figures --name e3
report/figures/e3_01_vlcsnap-00200.png
report/figures/e3_02_vlcsnap-00202.png
report/figures/e3_03_vlcsnap-00204.png
report/figures/e3_04_vlcsnap-2025-02-18-23h30m29s649.png
report/figures/e3_05_vlcsnap-2025-02-18-23h30m28s990.png
report/figures/e3_06_vlcsnap-2025-02-18-23h24m37s734.png
[PASS] evidence e0
[PASS] evidence e1
[PASS] evidence e2
[PASS] evidence e3
[PASS] evidence e4 — 2 file
[PASS] e1 meta
[PASS] e2 meta
[PASS] e3 meta
[PASS] dataset_version nhất quán — ['d10-21e4070301']
[PASS] khớp data/splits/meta.json
[FAIL] REPORT.md đã điền — còn chuỗi '<điền'
[PASS] git commits — 12/5
[PASS] git forbidden files

1 FAIL, 0 WARN


CompletedProcess(args=['/usr/bin/python3', 'scripts/check_submission.py'], returncode=1)

In [11]:
import zipfile
archive = Path("/kaggle/working/day10_results.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for pattern in ("report/evidence/*.json", "report/figures/*.png",
                    "work/*/run.json", "work/*_test.jsonl", "data/splits/meta.json"):
        for path in Path(".").glob(pattern):
            z.write(path, path.as_posix())
print("Download:", archive, f"({archive.stat().st_size / 1e6:.1f} MB)")
from IPython.display import FileLink, display
display(FileLink(str(archive)))


Download: /kaggle/working/day10_results.zip (4.0 MB)


/kaggle/working/day10_results.zip